# Forest Segmentation — SegFormer-B0 From Scratch (Colab)

Same pipeline as segformer_baseline_colab.ipynb, but the MiT-B0 encoder and
decode head are **randomly initialized** (no ImageNet pretrained weights).

Use this to compare against the pretrained baseline:
- pretrained notebook checkpoint: segformer_b0_baseline.pt
- this scratch checkpoint: segformer_b0_scratch.pt

Keep optimizer, splits, epochs, and metrics identical so the comparison is fair.
Expect lower Dice/IoU and/or slower convergence than the pretrained run.

**Before running:** Runtime → Change runtime type → GPU (T4).

## Step 0: Mount Google Drive

Upload your `Kalana` folder (with `images/` and `masks/` subfolders) to Google
Drive first, then mount it here. Adjust `DATA_ROOT` below to match wherever
you placed it in your Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers accelerate

## Step 1: Config + verify filename pairing

In [3]:
import os
from pathlib import Path

# ── Config ────────────────────────────────────────────────
# Adjust this path to wherever "Kalana" ended up inside your Drive.
# Example: "/content/drive/MyDrive/Kalana"
DATA_ROOT = Path("/content/drive/MyDrive/Kalana")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 20
LR = 6e-5
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42
CHECKPOINT_OUT = "/content/drive/MyDrive/segformer_b0_scratch.pt"
# ──────────────────────────────────────────────────────────

image_files = sorted(os.listdir(IMAGES_DIR))
mask_files = sorted(os.listdir(MASKS_DIR))

print(f"Found {len(image_files)} images")
print(f"Found {len(mask_files)} masks")


Found 5108 images
Found 5108 masks


In [4]:
def mask_name_to_image_name(mask_filename):
    """Converts '855_mask_01.jpg' -> '855_sat_01.jpg'."""
    return mask_filename.replace("_mask", "_sat")

# Full-dataset pairing check before we trust it on everything
missing = []
for m in mask_files:
    expected_image = mask_name_to_image_name(m)
    if expected_image not in image_files:
        missing.append((m, expected_image))

print(f"Total masks: {len(mask_files)}")
print(f"Missing matches: {len(missing)}")
if missing:
    print("First few missing pairs:", missing[:5])
assert len(missing) == 0, "Fix missing pairs before continuing."

Total masks: 5108
Missing matches: 0


## Step 2: Dataset class

Same logic verified locally — resize, normalize, binarize the mask.

In [5]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

class ForestSegDataset(Dataset):
    def __init__(self, mask_filenames, images_dir, masks_dir, img_size=256):
        self.mask_filenames = mask_filenames
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_size = img_size

        # ImageNet normalization stats (what SegFormer's pretrained backbone expects)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_fname = self.mask_filenames[idx]
        image_fname = mask_name_to_image_name(mask_fname)

        image = Image.open(self.images_dir / image_fname).convert("RGB")
        image = image.resize((self.img_size, self.img_size))

        mask = Image.open(self.masks_dir / mask_fname).convert("L")
        mask = mask.resize((self.img_size, self.img_size), resample=Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - self.mean) / self.std
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        # White (>127) = forest = 1, black = non-forest = 0
        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask).long()

        return image, mask

## Step 3: Train / val / test split

In [6]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_splits():
    all_files = sorted(mask_files)
    random.shuffle(all_files)

    n = len(all_files)
    n_val = int(n * VAL_SPLIT)
    n_test = int(n * TEST_SPLIT)

    val_files = all_files[:n_val]
    test_files = all_files[n_val:n_val + n_test]
    train_files = all_files[n_val + n_test:]
    return train_files, val_files, test_files

set_seed(SEED)
train_files, val_files, test_files = make_splits()
print(f"Train/Val/Test sizes: {len(train_files)}/{len(val_files)}/{len(test_files)}")

Train/Val/Test sizes: 3576/766/766


## Step 4: Model + Dice/IoU metric

Same SegFormer-B0 **architecture** as the pretrained baseline, but weights are
initialized from scratch (no HuggingFace pretrained download). Architecture matches MiT-B0:
depths [2,2,2,2], sr_ratios [8,4,2,1], etc.


In [7]:
from transformers import SegformerConfig, SegformerForSemanticSegmentation
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

# MiT-B0 stage config (SegFormer paper Table 6) - architecture only, no weights.
MIT_B0_CONFIG = dict(
    num_channels=3,
    num_encoder_blocks=4,
    depths=[2, 2, 2, 2],
    sr_ratios=[8, 4, 2, 1],
    hidden_sizes=[32, 64, 160, 256],
    num_attention_heads=[1, 2, 5, 8],
    patch_sizes=[7, 3, 3, 3],
    strides=[4, 2, 2, 2],
    mlp_ratios=[4, 4, 4, 4],
    decoder_hidden_size=256,
)

def build_model():
    """SegFormer-B0 with randomly initialized encoder + decode head."""
    cfg = SegformerConfig(
        num_labels=2,
        id2label={0: "non_forest", 1: "forest"},
        label2id={"non_forest": 0, "forest": 1},
        **MIT_B0_CONFIG,
    )
    model = SegformerForSemanticSegmentation(cfg)
    return model.to(device)

def dice_iou_score(pred_mask, true_mask, eps=1e-6):
    """pred_mask, true_mask: bool tensors, same shape. Dice/IoU for the
    'forest' (positive) class."""
    intersection = (pred_mask & true_mask).sum().float()
    pred_sum = pred_mask.sum().float()
    true_sum = true_mask.sum().float()
    union = pred_sum + true_sum - intersection

    dice = (2 * intersection + eps) / (pred_sum + true_sum + eps)
    iou = (intersection + eps) / (union + eps)
    return dice.item(), iou.item()

# Sanity: confirm we are NOT loading pretrained weights
_m = build_model()
n_params = sum(p.numel() for p in _m.parameters())
print(f"From-scratch SegFormer-B0 ready  |  params={n_params/1e6:.2f}M")
del _m


Running on: cuda
From-scratch SegFormer-B0 ready  |  params=3.71M


## Step 5: DataLoaders + training loop

This is the same loop verified locally on 10 images/5 epochs — now scaled up to the full dataset and 20 epochs on GPU.

In [8]:
from torch.utils.data import DataLoader
from tqdm import tqdm

train_ds = ForestSegDataset(train_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
val_ds = ForestSegDataset(val_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
test_ds = ForestSegDataset(test_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

In [9]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_dice, total_iou, n_batches = 0.0, 0.0, 0.0, 0

    with torch.set_grad_enabled(is_train):
        for images, masks in tqdm(loader, leave=False):
            images, masks = images.to(device), masks.to(device)

            outputs = model(pixel_values=images)
            logits = outputs.logits
            logits = F.interpolate(
                logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
            )

            loss = F.cross_entropy(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1).bool()
            dice, iou = dice_iou_score(preds, masks.bool())

            total_loss += loss.item()
            total_dice += dice
            total_iou += iou
            n_batches += 1

    return total_loss / n_batches, total_dice / n_batches, total_iou / n_batches

In [10]:
best_val_dice = 0.0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_dice, train_iou = run_epoch(model, train_loader, optimizer)
    val_loss, val_dice, val_iou = run_epoch(model, val_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} dice={train_dice:.4f} iou={train_iou:.4f} | "
        f"val_loss={val_loss:.4f} dice={val_dice:.4f} iou={val_iou:.4f}"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), CHECKPOINT_OUT)
        print(f"  -> saved new best checkpoint ({CHECKPOINT_OUT})")

Epoch 01 | train_loss=0.5515 dice=0.7890 iou=0.6594 | val_loss=0.4864 dice=0.8237 iou=0.7059
  -> saved new best checkpoint (/content/drive/MyDrive/segformer_b0_scratch.pt)


Epoch 02 | train_loss=0.5067 dice=0.8148 iou=0.6935 | val_loss=0.4566 dice=0.8417 iou=0.7312
  -> saved new best checkpoint (/content/drive/MyDrive/segformer_b0_scratch.pt)


Epoch 03 | train_loss=0.4875 dice=0.8247 iou=0.7079 | val_loss=0.4491 dice=0.8393 iou=0.7283


Epoch 04 | train_loss=0.4662 dice=0.8316 iou=0.7167 | val_loss=0.4823 dice=0.8419 iou=0.7316
  -> saved new best checkpoint (/content/drive/MyDrive/segformer_b0_scratch.pt)


Epoch 05 | train_loss=0.4519 dice=0.8360 iou=0.7239 | val_loss=0.4357 dice=0.8463 iou=0.7385
  -> saved new best checkpoint (/content/drive/MyDrive/segformer_b0_scratch.pt)


...

Epoch 16 | train_loss=0.2820 dice=0.9053 iou=0.8297 | val_loss=0.5107 dice=0.8229 iou=0.7053


Epoch 17 | train_loss=0.2785 dice=0.9070 iou=0.8322 | val_loss=0.5420 dice=0.8074 iou=0.6834


Epoch 18 | train_loss=0.2669 dice=0.9102 iou=0.8373 | val_loss=0.5500 dice=0.8244 iou=0.7075


Epoch 19 | train_loss=0.2659 dice=0.9095 iou=0.8372 | val_loss=0.5803 dice=0.8170 iou=0.6969


Epoch 20 | train_loss=0.2557 dice=0.9134 iou=0.8428 | val_loss=0.5359 dice=0.8342 iou=0.7207


## Step 6: Final test-set evaluation

Report this Dice/IoU next to the pretrained baseline (segformer_baseline_colab.ipynb)
to show how much ImageNet pretraining helps on this forest dataset.


In [11]:
model.load_state_dict(torch.load(CHECKPOINT_OUT))
test_loss, test_dice, test_iou = run_epoch(model, test_loader)
print(f"\nFINAL TEST RESULTS: dice={test_dice:.4f}  iou={test_iou:.4f}")


FINAL TEST RESULTS: dice=0.8451  iou=0.7368
